<a href="https://colab.research.google.com/github/mejia080902-bit/pymc-examples/blob/main/opcion_europea.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
INSTITUTO POLITÉCNICO NACIONAL
ESCUELA SUPERIOR DE FÍSICA Y MATEMÁTICAS

Seminario de modelación financiera
Profesor: Ramírez Reyes Francisco

PROYECTO: VALORACIÓN DE OPCIONES CALL EUROPEAS - MODELO BINOMIAL Y MONTE CARLO

Autor: Mejia Cedillo Diego


Este script implementa dos metodologías para calcular el precio de una opción financiera:
1. MODELO BINOMIAL (COX-ROSS-RUBINSTEIN): Calcula el precio exacto mediante inducción
   hacia atrás (backward induction), construyendo un árbol de precios donde el activo
   puede subir (u) o bajar (d) en cada paso temporal.

2. SIMULACIÓN DE MONTE CARLO: Estima el precio mediante la generación de miles de
   trayectorias aleatorias. Incluye técnicas avanzadas como:
   - Variables Antitéticas: Para reducir la varianza y mejorar la precisión.
   - Estimación por Trayectorias (Path) vs Distribución Directa (Binomial).
   - Cálculo de Error Estándar e Intervalos de Confianza al 95%.

El código permite comparar la solución analítica exacta frente a la aproximación
estocástica en un entorno neutral al riesgo.
"""

import numpy as np

def crr_ud(sigma: float, dt: float) -> tuple[float, float]:
    """
    Calcula los parámetros de Cox-Ross-Rubinstein a partir de la volatilidad.
    u = factor de subida, d = factor de bajada.

    Fórmulas:
    u = exp(sigma * sqrt(dt))
    d = exp(-sigma * sqrt(dt)) = 1/u
    """
    u = np.exp(sigma * np.sqrt(dt))
    d = np.exp(-sigma * np.sqrt(dt))
    return u, d

def risk_neutral_p(r_step: float, u: float, d: float) -> float:
    """
    Calcula la probabilidad neutral al riesgo para una tasa libre de riesgo por paso.
    (1 + r_step) es el factor de crecimiento bruto por paso.

    Verifica que se cumpla la condición de no arbitraje: d < (1 + r) < u.
    """
    if not (d < 1 + r_step < u):
        raise ValueError(f"Violación de no arbitraje: se requiere d < 1+r < u, se obtuvo d={d}, 1+r={1+r_step}, u={u}")
    return ((1 + r_step) - d) / (u - d)

def binomial_call_backward(S0: float, K: float, r_step: float, u: float, d: float, N: int) -> float:
    """
    Calcula el precio exacto de una opción Call Europea en un modelo binomial de N pasos
    mediante inducción hacia atrás (backward induction).
    """
    p = risk_neutral_p(r_step, u, d)
    disc = 1.0 / (1.0 + r_step) # Factor de descuento por paso

    # Precios finales del activo: S_N^{(j)} = S0 * u^j * d^(N-j), para j=0..N
    j = np.arange(N + 1)
    ST = S0 * (u ** j) * (d ** (N - j))

    # Valor de la opción al vencimiento (Payoff): max(ST - K, 0)
    V = np.maximum(ST - K, 0.0)

    # Inducción hacia atrás hasta llegar al tiempo t=0
    for _ in range(N):
        V = disc * (p * V[1:] + (1.0 - p) * V[:-1])

    return float(V[0])

def mc_binomial_call(
    S0: float,
    K: float,
    r_step: float,
    u: float,
    d: float,
    N: int,
    M: int = 200_000,
    seed: int = 123,
    method: str = "binomial",      # "binomial" (rápido) o "path" (trayectorias)
    antithetic: bool = True
) -> tuple[float, float, tuple[float, float]]:
    """
    Estimador de Monte Carlo para el precio de una opción Call Europea bajo la medida Q.

    Retorna:
      estimacion_precio, error_estandar, intervalo_confianza_95%
    """
    rng = np.random.default_rng(seed)
    p = risk_neutral_p(r_step, u, d)
    disc = (1.0 + r_step) ** (-N) # Descuento total sobre N pasos

    if method not in {"binomial", "path"}:
        raise ValueError("El método debe ser 'binomial' o 'path'")

    if method == "binomial":
        # J sigue una distribución Binomial(N, p) => ST = S0 * u^J * d^(N-J)
        # Nota: La técnica antitética es más natural en simulación por trayectorias ("path").
        J = rng.binomial(N, p, size=M)
        ST = S0 * (u ** J) * (d ** (N - J))
        payoff = np.maximum(ST - K, 0.0)

    else:
        # Simulación por trayectorias individuales con pasos Bernoulli
        if antithetic:
            # Variable antitética: usamos U y (1-U) para generar trayectorias correlacionadas negativamente
            U = rng.random((M, N))
            sube1 = (U < p)
            sube2 = ((1.0 - U) < p)

            J1 = sube1.sum(axis=1) # Número de subidas en trayectorias originales
            J2 = sube2.sum(axis=1) # Número de subidas en trayectorias antitéticas

            ST1 = S0 * (u ** J1) * (d ** (N - J1))
            ST2 = S0 * (u ** J2) * (d ** (N - J2))

            payoff1 = np.maximum(ST1 - K, 0.0)
            payoff2 = np.maximum(ST2 - K, 0.0)

            payoff = 0.5 * (payoff1 + payoff2)  # Promedio de pares antitéticos
        else:
            sube = (rng.random((M, N)) < p)
            J = sube.sum(axis=1)
            ST = S0 * (u ** J) * (d ** (N - J))
            payoff = np.maximum(ST - K, 0.0)

    # Cálculo de métricas estadísticas
    estimacion_precio = disc * payoff.mean()
    error_estandar = disc * payoff.std(ddof=1) / np.sqrt(M)
    ic95 = (estimacion_precio - 1.96 * error_estandar, estimacion_precio + 1.96 * error_estandar)

    return float(estimacion_precio), float(error_estandar), (float(ic95[0]), float(ic95[1]))


# -------------------------
# Ejemplo de uso
# -------------------------
if __name__ == "__main__":
    # Definición de variables del mercado
    S0, K = 100.0, 100.0  # Precio inicial y Strike
    r_step = 0.05         # Tasa por paso
    u, d = 1.2, 0.9       # Factores de subida y bajada
    N = 50                # Número de pasos

    # Cálculo exacto mediante el árbol binomial
    exacto = binomial_call_backward(S0, K, r_step, u, d, N)

    # Estimación mediante simulación de Monte Carlo
    mc, ee, ic = mc_binomial_call(S0, K, r_step, u, d, N, M=200_000, method="path", antithetic=True)

    # Impresión de resultados traducidos
    print(f"Precio Exacto (Binomial hacia atrás): {exacto:.4f}")
    print(f"Estimación de Monte Carlo: {mc:.4f}")
    print(f"Error Estándar: {ee:.6f}")
    print(f"Intervalo de Confianza 95%: ({ic[0]:.4f}, {ic[1]:.4f})")

Precio Exacto (Binomial hacia atrás): 91.3533
Estimación de Monte Carlo: 91.2090
Error Estándar: 0.164793
Intervalo de Confianza 95%: (90.8860, 91.5320)
